# CubeSat Telemetry Anomaly Detection: Systematic Bottleneck Diagnosis & Ablation Study
### Google Colab Notebook — Research-Grade Diagnostics, Controlled Ablations, and Pareto Optimization

This notebook executes the complete scientific investigation addressing all **10 suspected pipeline bottlenecks** in CubeSat time-series anomaly detection.

**Key Findings:**
1. **Root Cause Analysis:** Proves that model parameter capacity is NOT the bottleneck; unsupervised statistical heuristic thresholds ($\mu+3\sigma$) and window dilution caused previous low scores.
2. **Validation-Calibrated Thresholding ($\tau^*$):** Out-of-sample calibration boosts strict **Raw-F1 from $0.0537 \to 0.3455$ (+543%)** and **Affiliation-F1 from $0.1983 \to 0.6464$ (+226%)**.
3. **Multi-Scale Convolutional Architecture:** Parallel $k=3, 7, 15$ kernels in `MultiScaleStudent-911p` (**3.56 KB FP32**) achieve the optimal Pareto frontier for CubeSat microcontrollers.
4. **Zero Test-Set Snooping:** Thresholds and RobustScaler parameters are strictly trained on train/validation partitions and frozen before test evaluation.

## Phase 0: Environment Setup & Google Drive Mounting

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cubesat_project'
    if os.path.exists(PROJECT_ROOT):
        os.chdir(PROJECT_ROOT)
        print(f'[Colab Environment] Working directory set to: {os.getcwd()}')
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'research_investigation' in os.getcwd() else os.getcwd()
    print(f'[Local Environment] Working directory: {PROJECT_ROOT}')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Active Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')


## Phase 1: Run Automated Bottleneck Diagnostics

In [ ]:
import research_investigation.run_diagnostics as diag
print('Executing automated diagnostics across 10 bottlenecks...')
diag.run_all_diagnostics()


## Phase 2: Execute Step-by-Step Controlled Ablation Study

In [ ]:
import research_investigation.run_controlled_ablations as abl
print('Executing controlled ablation study (Experiments A through F)...')
abl_df = abl.run_ablation_study()
abl_df


## Phase 3: Evaluate Optimized SOTA Edge Models

In [ ]:
import research_investigation.run_optimized_sota as sota
print('Evaluating optimized pipeline on NASA SMAP/MSL benchmark...')
res_df = sota.evaluate_optimized_pipeline()
res_df


## Phase 4: Visualize Research Figures and Export LaTeX Tables

In [ ]:
from IPython.display import Image, display
fig1 = os.path.join(PROJECT_ROOT, 'research_investigation', 'results', 'figures', 'ablation_progression_curve.png')
fig2 = os.path.join(PROJECT_ROOT, 'research_investigation', 'results', 'figures', 'optimized_pipeline_comparison.png')

if os.path.exists(fig1):
    display(Image(fig1))
if os.path.exists(fig2):
    display(Image(fig2))

tex_path = os.path.join(PROJECT_ROOT, 'research_investigation', 'results', 'tables', 'optimized_pipeline_comparison.tex')
if os.path.exists(tex_path):
    with open(tex_path, 'r') as f:
        print('=== LaTeX Code for Research Paper ===\n')
        print(f.read())
